# 02 - Build Dimension Tables

## Overview

This notebook transforms the extracted MaStR generation units into the dimensional structures used by the Renewables-Climate Mart.

The previous notebook identified portfolio-related market actors and extracted their associated renewable electricity generation units from the German Market Master Data Register (MaStR). Building on these intermediate datasets, this notebook enriches the extracted units with additional geographic information and prepares the core dimensions required for subsequent generation modeling and analytical queries.

The transformation process includes the creation of hierarchical company, area, and energy source dimensions, as well as a consolidated asset dimension containing technical, geographic, and organizational attributes for each generation unit. In addition, a time dimension is generated to support temporal analysis in later stages of the pipeline.

The resulting dimension tables form the structural foundation of the Renewables-Climate Mart and are used in subsequent steps to estimate electricity generation and load the final data model into DuckDB.

## Input Data

This notebook expects the outputs generated by Notebook 01:

* `data/interim/portfolio_market_actors.csv`
* `data/interim/portfolio_generation_units.csv`

In addition, supplemental geographic reference data is used to enrich generation units with coordinates and transmission system operator (TSO) information.

## Outputs

The notebook produces the following dimension tables:

* `data/processed/dim_company_h.csv`
* `data/processed/dim_area_h.csv`
* `data/processed/dim_energy_source_h.csv`
* `data/processed/dim_asset.csv`
* `data/processed/dim_date.csv`


In [1]:
import pandas as pd

import re

import holidays

In [2]:
DATA_DIR = "../data"

RAW_GEO_DIR = f"{DATA_DIR}/raw/geo"
INTERIM_DIR = f"{DATA_DIR}/interim"
PROCESSED_DIR = f"{DATA_DIR}/processed"

## 1. Load and Enrich Portfolio Generation Units

This section loads the extracted portfolio generation units from the previous notebook and prepares the base dataset for building the asset dimension.

Only active generation units are retained. The MaStR commissioning date is converted into a datetime format, and the numeric MaStR energy carrier codes are mapped to readable energy source labels. The generation units are then joined with the validated portfolio market actors to add operator information.

The resulting dataset is reduced to the attributes required for the asset dimension, including energy source, operator, plant name, commissioning date, capacity, location, and coordinates. Since some MaStR entries do not contain coordinates, missing latitude and longitude values are identified and enriched using a supplemental city-level geographic reference table containing coordinates and transmission system operator information.

In [3]:
# Prepare extracted generation units for dimension building.

portfolio_generation_units_df = pd.read_csv(f"{INTERIM_DIR}/portfolio_generation_units.csv")

# Keep only units that are in operation ("In Betrieb")
portfolio_generation_units_df = portfolio_generation_units_df[portfolio_generation_units_df['EinheitBetriebsstatus']==35]

portfolio_generation_units_df['commissioning_date'] = pd.to_datetime(portfolio_generation_units_df['Inbetriebnahmedatum'], dayfirst=False)

energySource_dict = {2493: 'Biomass', 2497: 'Wind', 2495: 'Solar', 2498: 'Hydro'}
portfolio_generation_units_df['energy_source'] = portfolio_generation_units_df['Energietraeger'].map(energySource_dict)

portfolio_generation_units_df

,EinheitMastrNummer,NameStromerzeugungseinheit,EinheitBetriebsstatus,Inbetriebnahmedatum,Registrierungsdatum,Energietraeger,Bruttoleistung,Nettonennleistung,Postleitzahl,Ort,Bundesland,AnlagenbetreiberMastrNummer,DatumLetzteAktualisierung,Laengengrad,Breitengrad,commissioning_date,energy_source
0,SEE909081639072,BM-BHKW Heizwerk Kirchberg,35,2012-02-02,2019-02-01,2493,1189.00,1189.0,8107,Kirchberg,1413,ABR905920218880,2024-03-21T10:19:22.1843189,12.506653,50.619609,2012-02-02,Biomass
1,SEE909043722354,Heizkraftwerk Osterfeld,35,2009-01-14,2019-02-07,2493,1900.00,1900.0,6721,Osterfeld,1414,ABR996083385243,2019-05-28T08:26:38.0316692,11.934822,51.047045,2009-01-14,Biomass
2,SEE931543799284,"BGA Bitterfeld, Mühlenweg 1c, Anlage 1, BHKW",35,2007-06-19,2019-02-12,2493,625.00,625.0,6749,Bitterfeld-Wolfen,1414,ABR991967582008,2025-12-04T16:08:03.8315274,12.312000,51.630000,2007-06-19,Biomass
3,SEE904234294467,"BGA Kaakstedt, Ort Weiler 11, Anlage 1, BHKW",35,2006-09-05,2019-02-13,2493,526.00,526.0,17268,Gerswalde,1400,ABR991967582008,2024-02-26T16:13:04.0643562,13.779000,53.191000,2006-09-05,Biomass
4,SEE955362298346,"BGA Garlipp, Gewerbegebiet, Anlage 1, BHKW",35,2006-12-11,2019-02-13,2493,536.00,536.0,39628,Bismark,1414,ABR905920218880,2025-12-04T16:21:23.3134157,11.612176,52.639640,2006-12-11,Biomass
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,SEE997852162300,LHH_Vahrenwalder Bad,35,2025-08-01,2025-09-18,2495,40.50,30.0,30165,Hannover,1408,ABR930870189218,2025-09-18T13:12:47.4829509,NaN,NaN,2025-08-01,Solar
668,SEE935626747339,Verdion III,35,2025-09-10,2025-10-22,2495,1184.40,880.0,30539,Hannover,1408,ABR930870189218,2025-10-21T22:06:47.4203764,9.824000,52.329000,2025-09-10,Solar
675,SEE900745167801,Stadt Ronnenberg FW Weetzen,35,2025-11-03,2025-11-04,2495,26.24,25.0,30952,Ronnenberg,1408,ABR930870189218,2025-11-04T19:15:33.0924474,NaN,NaN,2025-11-03,Solar
676,SEE979470952174,Stadt Ronnenberg Sporthaus Benthe,35,2025-11-03,2025-11-04,2495,24.10,24.1,30952,Ronnenberg,1408,ABR930870189218,2025-11-04T20:03:59.8560665,NaN,NaN,2025-11-03,Solar


In [4]:
portfolio_market_actors_df = pd.read_csv(f"{INTERIM_DIR}/portfolio_market_actors.csv", index_col=None)

merge_df = pd.merge(
    portfolio_generation_units_df,
    portfolio_market_actors_df,
    left_on= 'AnlagenbetreiberMastrNummer',
    right_on='Mastr_Nummer',
    how = 'left')

merge_df = merge_df.rename(columns={
    'NameStromerzeugungseinheit': 'plant_name',
    'Firmenname': 'plant_operator',
    'Postleitzahl':'zipcode',
    'Ort_x':'city',
    'Breitengrad':'latitude',
    'Laengengrad':'longitude'

})

merge_df['generation_capacity_MW'] = merge_df['Nettonennleistung'] / 1000

merge_df = merge_df[['energy_source', 'plant_operator', 'plant_name', 'commissioning_date', 'generation_capacity_MW', 'zipcode', 'city', 'latitude', 'longitude']]

merge_df

,energy_source,plant_operator,plant_name,commissioning_date,generation_capacity_MW,zipcode,city,latitude,longitude
0,Biomass,Danpower Energie Service GmbH,BM-BHKW Heizwerk Kirchberg,2012-02-02,1.1890,8107,Kirchberg,50.619609,12.506653
1,Biomass,Danpower Pelletproduktion GmbH,Heizkraftwerk Osterfeld,2009-01-14,1.9000,6721,Osterfeld,51.047045,11.934822
2,Biomass,Danpower GmbH,"BGA Bitterfeld, Mühlenweg 1c, Anlage 1, BHKW",2007-06-19,0.6250,6749,Bitterfeld-Wolfen,51.630000,12.312000
3,Biomass,Danpower GmbH,"BGA Kaakstedt, Ort Weiler 11, Anlage 1, BHKW",2006-09-05,0.5260,17268,Gerswalde,53.191000,13.779000
4,Biomass,Danpower Energie Service GmbH,"BGA Garlipp, Gewerbegebiet, Anlage 1, BHKW",2006-12-11,0.5360,39628,Bismark,52.639640,11.612176
...,...,...,...,...,...,...,...,...,...
620,Solar,enercitySolution GmbH,LHH_Vahrenwalder Bad,2025-08-01,0.0300,30165,Hannover,NaN,NaN
621,Solar,enercitySolution GmbH,Verdion III,2025-09-10,0.8800,30539,Hannover,52.329000,9.824000
622,Solar,enercitySolution GmbH,Stadt Ronnenberg FW Weetzen,2025-11-03,0.0250,30952,Ronnenberg,NaN,NaN
623,Solar,enercitySolution GmbH,Stadt Ronnenberg Sporthaus Benthe,2025-11-03,0.0241,30952,Ronnenberg,NaN,NaN


In [5]:
# Inspect generation units for which coordinates could not be assigned directly.

missing_coordinates = merge_df[(merge_df['latitude'].isna() | merge_df['longitude'].isna())].index.tolist()
merge_df.iloc[missing_coordinates][['zipcode','city', 'latitude', 'longitude']].drop_duplicates()

,zipcode,city,latitude,longitude
54,48455,Bad Bentheim,NaN,NaN
55,30559,Hannover,NaN,NaN
56,22559,Hamburg,NaN,NaN
58,22885,Barsbüttel,NaN,NaN
65,30880,Laatzen,NaN,NaN
66,30966,Hemmingen,NaN,NaN
70,30519,Hannover,NaN,NaN
77,30453,Hannover,NaN,NaN
80,30161,Hannover,NaN,NaN
81,30827,Garbsen,NaN,NaN


In [6]:
geo_df = pd.read_csv(f"{RAW_GEO_DIR}/cities_with_tso_and_coords.csv", dtype={'zipcode': int})

merge_df = pd.merge(
    merge_df, 
    geo_df, 
    on=['zipcode','city'], 
    how='left'
)

merge_df['latitude'] = merge_df['latitude'].fillna(merge_df['latitude_cen'])
merge_df['longitude'] = merge_df['longitude'].fillna(merge_df['longitude_cen'])

## 2. Build Company Hierarchy Dimension

This section constructs the company hierarchy dimension (`dim_company_h`) used by the Renewables-Climate Mart.

The process begins by identifying which portfolio companies are associated with extracted generation units. Using the list of directly owned enercity subsidiaries as a reference, companies are classified into direct and indirect subsidiaries. For indirect subsidiaries, parent companies are inferred through address matching based on the validated market actor dataset. Remaining unresolved cases are reviewed and corrected manually.

To support the public release of the dataset, company and parent names are subsequently pseudonymized using the Electricville branding while preserving the underlying organizational structure.

In [7]:
companies = merge_df['plant_operator'].drop_duplicates().tolist()

# Extracted from the enercity annual report 2025
directly_owned = ["Danpower GmbH",
    "Elektroanlagenbau Kammeyer GmbH",
    "enercity Contracting GmbH",
    "enercity digital GmbH",
    "enercity Erneuerbare GmbH",
    "enercity Fachpartner-Holding GmbH",
    "enercity Flughafen Netz GmbH",
    "enercity Grid Partner GmbH",
    "enercity Speichervermarktungsgesellschaft mbH",
    "enercitySolution GmbH",
    "KLH Tiefwerk Holding GmbH",
    "The Mother Nature GmbH",
    "enercity Netz GmbH",
    "GKH-Gemeinschaftskraftwerk Hannover GmbH",
    "GHG- Gasspeicher Hannover GmbH",
    "Rockethome Climate Solutions GmbH",
    "GHG- Gasspeicher Hannover GbR",
    "htp GmbH",
    "Gasnetzgesellschaft Laatzen-Nord mbH",
    "Gasnetzgesellschaft Seelze Gmbh ＆ Co. KG",
    "Netzgesellschaft Laatzen GmbH ＆ Co. KG",
    "Netzverwaltungsgesellschaft Laatzen mbH",
    "TRIGIS NET GmbH",
    "Stadtwerke Wunstorf GmbH ＆ Co. KG",
    "Stadtwerke Wunstorf Verwaltungs GmbH",
    "ev-pay GmbH",
    "ROCKETHOME GmbH",
    "Stadtwerke Garbsen GmbH",
    "Autostrom plus GmbH",
    "Thüga Holding GmbH ＆ Co. KGaA",]

set(companies) & set(directly_owned)


{'Danpower GmbH',
 'Stadtwerke Wunstorf GmbH ＆ Co. KG',
 'enercity Contracting GmbH',
 'enercity Erneuerbare GmbH',
 'enercitySolution GmbH',
 'htp GmbH'}

In [8]:
# Addresses of the directly owned companies
directlyOwned_df = portfolio_market_actors_df[portfolio_market_actors_df['Firmenname'].isin(directly_owned)].reset_index(drop=True)
company_lookup = directlyOwned_df.set_index(['Adresse'])['Firmenname'].to_dict()

def get_parentNLevel(name):
    top_level = 'enercity AG'

    if name =='enercity AG':
        return top_level, 'Holding'
    
    if name in directly_owned:
        return top_level, 'direct Subsidiary'
    
    else: 
        address_series = portfolio_market_actors_df[portfolio_market_actors_df['Firmenname']==name].Adresse
        if not address_series.empty:
            # Take the address and lookup the parent_company
            address = address_series.iloc[0] 
            parent = company_lookup.get(address.strip(), "Unknown Parent")
        else:
            parent = "Address Not Found"
        
    return parent, 'indirect Subsidiary'

# Iterate through all companies with identified renewable generation units    
hierarchy_data = []
for i, name in enumerate(sorted(companies)):
    parent, top = get_parentNLevel(name)
    hierarchy_data.append({
        "plant_operator": name,
        "parent_company": parent,
        "company_level": top
    })

dim_company_h = pd.DataFrame(hierarchy_data)

# Name normalization by lowering and removing white spaces
dim_company_h['name_normalized'] = dim_company_h['plant_operator'].str.strip().str.lower()

# Remove duplicates from writing variations
dim_company_h = dim_company_h.drop_duplicates(subset=['name_normalized'], keep='last')

# Delete helper columns
dim_company_h = dim_company_h.drop(columns=['name_normalized'])

dim_company_h = dim_company_h.reset_index(drop=True)
dim_company_h

,plant_operator,parent_company,company_level
0,"""Windpark Norderland GmbH ＆ Co. KG, Holtriemer...",enercity Erneuerbare GmbH,indirect Subsidiary
1,BEH Bioenergie Hannover GmbH,Danpower GmbH,indirect Subsidiary
2,Carpe Ventos Wiesmoor III GmbH ＆ Co. KG,enercity Erneuerbare GmbH,indirect Subsidiary
3,Carpe Ventos Wiesmoor IV GmbH ＆ Co. KG,enercity Erneuerbare GmbH,indirect Subsidiary
4,Carpe Ventos Wiesmoor VII GmbH ＆ Co. KG,enercity Erneuerbare GmbH,indirect Subsidiary
...,...,...,...
85,enercity Windpark Portfolio GmbH ＆ Co. KG,Unknown Parent,indirect Subsidiary
86,enercity Windpark Portfolio II GmbH,enercity Erneuerbare GmbH,indirect Subsidiary
87,enercity Windpark Schipkau GmbH ＆ Co. KG,Unknown Parent,indirect Subsidiary
88,enercitySolution GmbH,enercity AG,direct Subsidiary


In [9]:
# Validate that all companies could be assigned to a parent company.

dim_company_h[dim_company_h['parent_company']=='Unknown Parent']

,plant_operator,parent_company,company_level
14,EBV Windpark Almstedt-Breinum GmbH ＆ Co. Betri...,Unknown Parent,indirect Subsidiary
15,ELW Energieversorgung Leinefelde-Worbis GmbH,Unknown Parent,indirect Subsidiary
16,Fiba Energieservice GmbH,Unknown Parent,indirect Subsidiary
22,IEW Biogaspark Wolgast GmbH,Unknown Parent,indirect Subsidiary
66,Wärmeversorgung Wolgast GmbH,Unknown Parent,indirect Subsidiary
80,enercity Windpark Groß Eilstorf GmbH,Unknown Parent,indirect Subsidiary
82,enercity Windpark Klettwitz GmbH ＆ Co. KG,Unknown Parent,indirect Subsidiary
85,enercity Windpark Portfolio GmbH ＆ Co. KG,Unknown Parent,indirect Subsidiary
87,enercity Windpark Schipkau GmbH ＆ Co. KG,Unknown Parent,indirect Subsidiary


In [10]:
# Manually resolve companies whose parent assignment could not be inferred automatically.

correction_dict = {
    14: 'enercity Erneuerbare GmbH',
    15: 'Danpower GmbH',
    16: 'Danpower GmbH',
    22: 'Danpower GmbH',
    66: 'Danpower GmbH',
    80: 'enercity Erneuerbare GmbH',
    82: 'enercity Erneuerbare GmbH',
    85: 'enercity Erneuerbare GmbH',
    87: 'enercity Erneuerbare GmbH',
}

for idx, values in correction_dict.items():
    dim_company_h.loc[idx, 'parent_company'] = values

In [11]:
# Transforms real-world names into pseudonymized 'electricville' branding. Maintains suffixes and project-specific identifiers for context.

def apply_electricville_branding(text):
    # Define the mapping
    mapping = {
        'enercity': 'electricville',
        'Danpower': 'Lancepower',
        'Stadtwerke': 'Gemeindewerk',
        'htp GmbH': 'pth GmbH',
        'Bioenergie': 'Biomasse',
        'Energie': 'Strom',
        'park':'farm',
        'Wärme':'Heiz',
        'Carpe':'Utere',
        'Freesen-Wind': 'Freesen-Breeze',
        'Freesenwind': 'Freesen-Breeze',
        'Norderland': 'Nordisch',
        'Innerstetal': 'Interstellar',
        'WEA': 'WKA',
        'Windkraftanlage': 'Windenergieanlage'

    }

    # Apply each transformation
    for pattern, replacement in mapping.items():
        text = re.compile(pattern, re.IGNORECASE).sub(replacement, text)
    
    return text

In [12]:
# 1. Update Company Dimension
dim_company_h['company_name'] = dim_company_h['plant_operator'].apply(apply_electricville_branding).str.replace('"','')
dim_company_h['parent_company'] = dim_company_h['parent_company'].apply(apply_electricville_branding)

# 2. Set company_id
dim_company_h = dim_company_h.sort_values(["company_level","parent_company","company_name"]).reset_index(drop=True)
dim_company_h['company_id'] = range(1, len(dim_company_h) + 1)
cols = ['company_id'] + [c for c in dim_company_h.columns if c != 'company_id']
dim_company_h = dim_company_h[cols]
dim_company_h

,company_id,plant_operator,parent_company,company_level,company_name
0,1,enercity AG,electricville AG,Holding,electricville AG
1,2,Stadtwerke Wunstorf GmbH ＆ Co. KG,electricville AG,direct Subsidiary,Gemeindewerk Wunstorf GmbH ＆ Co. KG
2,3,Danpower GmbH,electricville AG,direct Subsidiary,Lancepower GmbH
3,4,enercity Contracting GmbH,electricville AG,direct Subsidiary,electricville Contracting GmbH
4,5,enercity Erneuerbare GmbH,electricville AG,direct Subsidiary,electricville Erneuerbare GmbH
...,...,...,...,...,...
85,86,enercity Windpark Lemwerder GmbH,electricville Erneuerbare GmbH,indirect Subsidiary,electricville Windfarm Lemwerder GmbH
86,87,enercity Windpark Münstedt II GmbH ＆ Co. KG,electricville Erneuerbare GmbH,indirect Subsidiary,electricville Windfarm Münstedt II GmbH ＆ Co. KG
87,88,enercity Windpark Portfolio GmbH ＆ Co. KG,electricville Erneuerbare GmbH,indirect Subsidiary,electricville Windfarm Portfolio GmbH ＆ Co. KG
88,89,enercity Windpark Portfolio II GmbH,electricville Erneuerbare GmbH,indirect Subsidiary,electricville Windfarm Portfolio II GmbH


## 3. Build Supporting Dimensions

This section creates supporting dimensions used by the asset dimension.

The geographic area dimension (`dim_area_h`) is derived from the enriched generation unit dataset by grouping location-related attributes such as postal code, city, state, TSO region, energy source, and plant operator. Representative coordinates are calculated for each group using the mean latitude and longitude of the associated generation units.

The energy source dimension (`dim_energy_source_h`) standardizes the renewable energy source categories used throughout the mart and groups them under the renewable energy domain.

Both dimensions are later referenced by the asset dimension to avoid repeating descriptive attributes directly in the asset table.

In [13]:
# Build geographic area dimension.

dim_area_h = merge_df.groupby(['zipcode','city','state','tso_region','energy_source', 'plant_operator'])[['latitude', 'longitude']].mean()

dim_area_h = dim_area_h.sort_values(['state','city']).reset_index()

# area_id
dim_area_h['area_id'] = range(1, len(dim_area_h) + 1)
cols = ['area_id'] + [c for c in dim_area_h.columns if c != 'area_id']
dim_area_h = dim_area_h[cols]

dim_area_h

,area_id,zipcode,city,state,tso_region,energy_source,plant_operator,latitude,longitude
0,1,79115,Freiburg,Baden-Württemberg,TransnetBW,Solar,enercitySolution GmbH,47.979300,7.822000
1,2,76187,Karlsruhe,Baden-Württemberg,TransnetBW,Solar,enercitySolution GmbH,49.019815,8.371491
2,3,73230,Kirchheim,Baden-Württemberg,TransnetBW,Biomass,Danpower GmbH,48.615626,9.475712
3,4,71638,Ludwigsburg,Baden-Württemberg,TransnetBW,Solar,enercity AG,48.893692,9.225544
4,5,88709,Meersburg,Baden-Württemberg,TransnetBW,Biomass,enercity Contracting GmbH,47.704913,9.271325
...,...,...,...,...,...,...,...,...,...
236,237,7586,Bad Köstritz,Thüringen,50Hertz,Biomass,Danpower Biomasse GmbH,50.930958,12.036299
237,238,7356,Bad Lobenstein,Thüringen,50Hertz,Biomass,Danpower GmbH,50.462087,11.636383
238,239,7957,Langenwetzendorf,Thüringen,50Hertz,Biomass,Danpower GmbH,50.689310,12.113784
239,240,37327,Leinefelde-Worbis,Thüringen,50Hertz,Biomass,ELW Energieversorgung Leinefelde-Worbis GmbH,51.382099,10.339433


In [14]:
dim_energy_source_h = pd.DataFrame()
dim_energy_source_h['energy_source_name'] = merge_df['energy_source'].drop_duplicates().reset_index(drop=True)
dim_energy_source_h['energy_source_group'] = 'Renewable'

dim_energy_source_h['energy_source_id'] = range(1, len(dim_energy_source_h) + 1)
cols = ['energy_source_id'] + [c for c in dim_energy_source_h.columns if c != 'energy_source_id']
dim_energy_source_h = dim_energy_source_h[cols]
dim_energy_source_h

,energy_source_id,energy_source_name,energy_source_group
0,1,Biomass,Renewable
1,2,Wind,Renewable
2,3,Hydro,Renewable
3,4,Solar,Renewable


## 4. Build Asset Dimension

This section creates the central asset dimension (`dim_asset`) from the enriched generation unit dataset.

The previously created company, area, and energy source dimensions are linked to each generation unit by assigning the corresponding dimension identifiers. The commissioning date is also converted into a date key so it can later reference the time dimension.

The asset dimension retains the attributes required for analysis and generation modeling, including company, area, energy source, commissioning date, asset name, and installed generation capacity. Asset names are pseudonymized using the same Electricville branding applied to the company dimension.

In [15]:
# area_id
area_lookup = dim_area_h.set_index(['zipcode','city', 'energy_source', 'plant_operator'])['area_id'].to_dict()

merge_df['area_id'] = [
    area_lookup.get((z, c, e, o)) 
    for z, c, e, o in zip(merge_df['zipcode'], merge_df['city'], merge_df['energy_source'], merge_df['plant_operator'])
]

# company_id
dim_company_h['comNam_lower'] = dim_company_h['plant_operator'].str.strip().str.lower()
company_lookup = dim_company_h.set_index(['comNam_lower'])['company_id'].to_dict()
dim_company_h = dim_company_h.drop(columns=['comNam_lower'])

merge_df['company_id'] = merge_df['plant_operator'].str.strip().str.lower().map(company_lookup)

# energy_source_id
merge_df = merge_df.merge(dim_energy_source_h, left_on='energy_source', right_on='energy_source_name')

# commissioning_date_id
merge_df['commissioning_date_id'] = merge_df['commissioning_date'].dt.strftime('%Y%m%d').astype(int)


In [16]:
asset_cols = ['company_id','area_id','energy_source_id','commissioning_date_id','asset_name', 'generation_capacity_MW']

merge_df = merge_df.rename( columns={
    'plant_name': 'asset_name'
})

dim_asset = merge_df[asset_cols].copy()

# Create a sequential surrogate key.
dim_asset['asset_id'] = range(1, len(dim_asset) + 1)
cols = ['asset_id'] + asset_cols
dim_asset = dim_asset[cols]

dim_asset['asset_name'] = dim_asset['asset_name'].apply(apply_electricville_branding)

dim_asset

,asset_id,company_id,area_id,energy_source_id,commissioning_date_id,asset_name,generation_capacity_MW
0,1,15,203,1,20120202,BM-BHKW Heizwerk Kirchberg,1.1890
1,2,14,221,1,20090114,Heizkraftwerk Osterfeld,1.9000
2,3,3,215,1,20070619,"BGA Bitterfeld, Mühlenweg 1c, Anlage 1, BHKW",0.6250
3,4,3,19,1,20060905,"BGA Kaakstedt, Ort Weiler 11, Anlage 1, BHKW",0.5260
4,5,15,213,1,20061211,"BGA Garlipp, Gewerbegebiet, Anlage 1, BHKW",0.5360
...,...,...,...,...,...,...,...
620,621,6,80,4,20250801,LHH_Vahrenwalder Bad,0.0300
621,622,6,108,4,20250910,Verdion III,0.8800
622,623,6,140,4,20251103,Stadt Ronnenberg FW Weetzen,0.0250
623,624,6,140,4,20251103,Stadt Ronnenberg Sporthaus Benthe,0.0241


## 5. Build Date Dimension

This section creates the date dimension (`dim_date`) used for temporal analysis in the Renewables-Climate Mart.

The date range starts with the earliest commissioning date found in the asset dimension and extends to the end of the analysis period on 2025-12-31. For each date, standard calendar attributes are generated, including weekday, ISO week, month, quarter, year, season, and weekend status.

German public holidays for Lower Saxony are added using the `holidays` package. Since the required date range starts before German reunification, selected historical holidays are patched manually for dates before 1991. Based on weekend and holiday information, a workday indicator is derived.

In [17]:
# 1. Identify the required range
# Convert your date_ids back to timestamps to find min/max
earliest_date = pd.to_datetime(min(dim_asset['commissioning_date_id']), format='%Y%m%d')
latest_date = pd.to_datetime('2025-12-31')

# 2. Generate the complete date range required.
dim_date = pd.DataFrame()
dim_date['date'] = pd.date_range(start=earliest_date, end=latest_date)

# 3. Standard Attributes
dim_date['date_id'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['day_of_year'] = dim_date['date'].dt.day_of_year
dim_date['weekday'] = dim_date['date'].dt.day_name()
dim_date['weekday_num'] = dim_date['date'].dt.isocalendar().day
dim_date['week_iso'] = dim_date['date'].dt.strftime('W%V')
dim_date['month_name'] = dim_date['date'].dt.month_name()
dim_date['month'] = dim_date['date'].dt.month
dim_date['quarter'] = 'Q' + dim_date['date'].dt.quarter.astype(str)
dim_date['year'] = dim_date['date'].dt.year
dim_date['year_iso'] = dim_date['date'].dt.isocalendar().year

# 4. Seasons
month_to_season = {
    1: 'Winter', 2: 'Winter', 3: 'Spring',
    4: 'Spring', 5: 'Spring', 6: 'Summer',
    7: 'Summer', 8: 'Summer', 9: 'Autumn',
    10: 'Autumn', 11: 'Autumn', 12: 'Winter'
}
dim_date['season'] = dim_date['month'].map(month_to_season)
dim_date['is_weekend'] = dim_date['date'].dt.dayofweek >= 5

In [18]:
# 5. Holidays

# Standard Library call (covers 1991-2025)
de_holidays = holidays.Germany(subdiv='NI', years=range(1928, 2026))

def get_historical_holiday(row):
    if row['date'] in de_holidays:
        return True
    
    # Patch for pre-1991
    if row['year'] < 1991:
        d, m = row['date'].day, row['date'].month

        fixed_holidays = [
            (1, 1),   # New Year
            (1, 5),   # Labor Day (since 1933/post-war)
            (17, 6),  # Old German Unity Day (WEST - until 1990)
            (25, 12), # 1. Christmas Day
            (26, 12)  # 2. Christmas Day
        ]
        if (d, m) in fixed_holidays:
            return True
            
    return False

# Apply to DataFrame
dim_date['is_holiday'] = dim_date.apply(get_historical_holiday, axis=1)

# Define workdays
dim_date['is_workday'] = ~dim_date['is_weekend'] & ~dim_date['is_holiday']

# Reorder
cols = ['date_id'] + [c for c in dim_date.columns if c != 'date_id']
dim_date = dim_date[cols]

## 6. Export Dimension Tables

This section performs the final preparation and export of the dimension tables created throughout the notebook.

Only the attributes required for the final dimensional model are retained, and each dimension is exported to the processed data layer. These tables form the structural foundation of the Renewables-Climate Mart and are used by subsequent notebooks to estimate electricity generation and construct the final DuckDB database.

In [19]:
dim_company_h = dim_company_h[['company_id', 'company_name', 'parent_company', 'company_level']]
dim_area_h = dim_area_h[['area_id','zipcode','city','state','tso_region','latitude','longitude']]


In [20]:
print(f"dim_company_h: {len(dim_company_h):,}")
print(f"dim_area_h: {len(dim_area_h):,}")
print(f"dim_energy_source_h: {len(dim_energy_source_h):,}")
print(f"dim_asset: {len(dim_asset):,}")
print(f"dim_date: {len(dim_date):,}")

dim_company_h: 90
dim_area_h: 241
dim_energy_source_h: 4
dim_asset: 625
dim_date: 35,643


In [21]:
dim_company_h.to_csv(f"{PROCESSED_DIR}/dim_company_h.csv", index=False)
dim_area_h.to_csv(f"{PROCESSED_DIR}/dim_area_h.csv", index=False)
dim_energy_source_h.to_csv(f"{PROCESSED_DIR}/dim_energy_source_h.csv",index=False)
dim_asset.to_csv(f"{PROCESSED_DIR}/dim_asset.csv", index=False)
dim_date.to_csv(f"{PROCESSED_DIR}/dim_date.csv", index=False)